# Job-Resume Matching — Notebook 03: Scoring Engine
Score your resume against every job using:
- Degree rule (20%)
- Major rule (20%)
- Sentence-BERT semantic skills similarity (60%)

In [1]:
import pandas as pd
import numpy as np
import ast, json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
print('Libraries loaded.')

Libraries loaded.


In [2]:
# Configuration
DEGREE_LEVELS = {'bachelor': 1, 'master': 2, 'phd': 3}

MAJOR_RELATIONS = {
    'computer_science':       ['computer_science','software_engineering','data_science',
                               'information_technology','artificial_intelligence'],
    'data_science':           ['data_science','statistics','mathematics',
                               'computer_science','artificial_intelligence'],
    'software_engineering':   ['software_engineering','computer_science','information_technology'],
    'electrical_engineering': ['electrical_engineering','computer_science'],
    'statistics':             ['statistics','mathematics','data_science'],
    'mathematics':            ['mathematics','statistics'],
    'information_technology': ['information_technology','computer_science','software_engineering'],
    'artificial_intelligence':['artificial_intelligence','computer_science','data_science'],
    'business':               ['business'],
}

WEIGHT_DEGREE = 0.20
WEIGHT_MAJOR  = 0.20
WEIGHT_SKILLS = 0.60

def safe_list(val):
    if isinstance(val, list): return val
    if pd.isna(val) or str(val).strip() in ('','[]','None'): return []
    try:    return ast.literal_eval(val)
    except: return []

print('Configuration loaded.')

Configuration loaded.


In [3]:
# Scoring functions
def degree_score(resume_degree, required_degree):
    if not required_degree or required_degree == '': return 1.0
    if not resume_degree   or resume_degree   == '': return 0.0
    req = DEGREE_LEVELS.get(required_degree, 0)
    res = DEGREE_LEVELS.get(resume_degree,   0)
    if res >= req:     return 1.0
    if res == req - 1: return 0.5
    return 0.0

def major_score(resume_majors, required_majors):
    if not required_majors: return 1.0
    if not resume_majors:   return 0.0
    acceptable = set()
    for m in required_majors:
        acceptable.update(MAJOR_RELATIONS.get(m, [m]))
    return 1.0 if any(m in acceptable for m in resume_majors) else 0.0

print('Scoring functions ready.')
print('Test degree (master vs master):', degree_score('master','master'))
print('Test major  (data_science vs statistics):', major_score(['data_science'],['statistics']))

Scoring functions ready.
Test degree (master vs master): 1.0
Test major  (data_science vs statistics): 1.0


In [4]:
# Load Sentence-BERT
print('Loading Sentence-BERT (first run downloads ~400MB)...')
sbert = SentenceTransformer('bert-base-nli-mean-tokens')
print('Model ready!\n')

def skills_score(resume_skills, job_skills):
    if not resume_skills or not job_skills: return 0.0
    r = np.mean(sbert.encode(resume_skills), axis=0, keepdims=True)
    j = np.mean(sbert.encode(job_skills),    axis=0, keepdims=True)
    return float(np.clip(cosine_similarity(r, j)[0][0], 0, 1))

Loading Sentence-BERT (first run downloads ~400MB)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/bert-base-nli-mean-tokens
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready!



In [5]:
# Load resume and jobs
with open('data/resume_info.json') as f:
    resume_info = json.load(f)

jobs_df = pd.read_csv('data/jobs_extracted.csv')

print('Your resume profile:')
print(f'  Degree : {resume_info["degree"]}')
print(f'  Majors : {resume_info["majors"]}')
print(f'  Skills : {len(resume_info["skills"])} detected')
print(f'\nJobs to score: {len(jobs_df)}')

Your resume profile:
  Degree : None
  Majors : ['computer_science', 'data_science', 'software_engineering', 'artificial_intelligence', 'business']
  Skills : 14 detected

Jobs to score: 352


In [6]:
# Score every job
r_degree = resume_info['degree']
r_majors = resume_info['majors']
r_skills = resume_info['skills']

results = []
for i, (_, job) in enumerate(jobs_df.iterrows(), 1):
    j_skills = safe_list(job['req_skills'])
    j_degree = job['req_degree'] if str(job['req_degree']) != 'nan' else ''
    j_majors = safe_list(job['req_majors'])

    d     = degree_score(r_degree, j_degree)
    m     = major_score(r_majors, j_majors)
    s     = skills_score(r_skills, j_skills)
    final = WEIGHT_DEGREE*d + WEIGHT_MAJOR*m + WEIGHT_SKILLS*s

    results.append({
        'company':      str(job['company']),
        'title':        str(job['title']),
        'source':       str(job.get('source','Unknown')),
        'degree_score': round(d,     4),
        'major_score':  round(m,     4),
        'skills_score': round(s,     4),
        'final_score':  round(final, 4),
        'job_skills':   str(j_skills),
        'your_skills':  str(r_skills),
    })
    print(f'\r  {i}/{len(jobs_df)}: {job["company"]} — {job["title"]}', end='')

results_df = (pd.DataFrame(results)
              .sort_values('final_score', ascending=False)
              .reset_index(drop=True))
results_df['rank'] = results_df.index + 1
results_df.to_csv('data/results.csv', index=False)
print(f'\n\nDone! Saved to data/results.csv\n')
display(results_df[['rank','source','company','title','degree_score','major_score','skills_score','final_score']])

  158/352: Tecolote Research Developerpere Engineering, Back End, Python, Spring Boot, AWS)ingeinsurance) - 100% Remote
  159/352: University of Maryland Medical System
  160/352: KnowBe4ata Scientist
  161/352: PNNLntist
  162/352: Affinity Solutions
  163/352: CyrusOnet
  164/352: ClearOne Advantage
  165/352: Logic20/20
  166/352: <intent>t
  167/352: Wishntist
  168/352: ManTechst
  169/352: Walmartst
  170/352: YeslerScientist - Technology
  171/352: Takeda Pharmaceuticals
  172/352: Audiblest
  173/352: h2o.aier I
  174/352: NunaData Scientist
  175/352: Pinnacol Assurance Data Analytics
  176/352: Porchtist
  177/352: Health IQ
  178/352: Truckstop.comist / Machine Learning
  179/352: SMC 3tist - Quantitative
  180/352: Marsntist
  181/352: Novettast
  182/352: Pfizerist
  183/352: First Tech Federal Credit Union
  184/352: The Hanover Insurance Group
  185/352: Pfizerata Analyst
  186/352: Amrockta Scientist
  187/352: Novartist
  188/352: Juniper Networksine Learning Expert
  

,rank,source,company,title,degree_score,major_score,skills_score,final_score
0,1,LinkedIn,Qentelli,Software Engineer in Test Automation,1.0,1.0,0.9768,0.9861
1,2,Glassdoor,goTRG\n4.2,Data Scientist,1.0,1.0,0.9754,0.9852
2,3,Glassdoor,Stratagem Group\n4.3,Machine Learning Engineer,1.0,1.0,0.9747,0.9848
3,4,LinkedIn,"Infotech Spectrum Inc,",SRE/DevOps Support Engineer (W2),1.0,1.0,0.9709,0.9826
4,5,Glassdoor,Maven Wave Partners\n4.4,Data Scientist,1.0,1.0,0.9705,0.9823
...,...,...,...,...,...,...,...,...
347,348,Glassdoor,Audible\n3.6,Data Engineer I,1.0,1.0,0.0000,0.4000
348,349,LinkedIn,Unknown,Senior Data Engineer/Analyst - Full Time,1.0,1.0,0.0000,0.4000
349,350,Glassdoor,Beck's Hybrids\n4.6,Ag Data Scientist,1.0,1.0,0.0000,0.4000
350,351,Glassdoor,Gensco\n4.4,Data Analyst,0.0,1.0,0.0000,0.2000


In [7]:
# Skills gap for top 3
your_lower = set(s.lower() for s in r_skills)

print('=' * 62)
print('  SKILLS GAP — TOP 3 MATCHES')
print('=' * 62)

for _, row in results_df.head(3).iterrows():
    import ast
    j_skills = ast.literal_eval(row['job_skills']) if isinstance(row['job_skills'], str) else []
    have    = [s for s in j_skills if s.lower() in your_lower]
    missing = [s for s in j_skills if s.lower() not in your_lower]

    print(f"\n#{int(row['rank'])}  [{row['source']}] {row['company']} — {row['title']}")
    print(f"    Score   : {row['final_score']:.4f}")
    print(f"    You have: {', '.join(have) if have else 'none detected'}")
    print(f"    Missing : {', '.join(missing) if missing else 'great fit!'}")

  SKILLS GAP — TOP 3 MATCHES

#1  [LinkedIn] Qentelli — Software Engineer in Test Automation
    Score   : 0.9861
    You have: Python, Java, JavaScript, SQL, Git
    Missing : C#, Tableau, Jenkins, CI/CD, GraphQL, Agile, communication

#2  [Glassdoor] goTRG
4.2 — Data Scientist
    Score   : 0.9852
    You have: Python, Java, JavaScript, machine learning, SQL
    Missing : R, Scala, data visualization, regression, statistics, Apache Spark, Hadoop, research

#3  [Glassdoor] Stratagem Group
4.3 — Machine Learning Engineer
    Score   : 0.9848
    You have: Python, Java, machine learning
    Missing : C++, TensorFlow, deep learning, neural networks, statistics, Hadoop, Docker, AWS, PostgreSQL, Agile
